# Deep Learning Transformer Training for Fake News & Hate Speech Detection
### Optimized for NVIDIA GeForce RTX 3050 (4GB VRAM) & Google Colab GPU

This notebook provides a complete fine-tuning pipeline for state-of-the-art Transformer models on the Indian Fake News & Hate Speech corpora.

#### Key Highlights for 4GB VRAM:
- **Mixed Precision (FP16)**: Halves memory usage without losing precision.
- **Gradient Accumulation**: Uses batch size 8 with 2-step accumulation for an effective batch size of 16.
- **Supported Models**:
  - `microsoft/deberta-v3-small` (Recommended for fast, high-accuracy inference)
  - `roberta-base`
  - `bert-base-uncased`
  - `ai4bharat/indic-bert`

In [ ]:
# Step 1: Verify CUDA Hardware & GPU Diagnostics
import torch

print("=" * 60)
print("SYSTEM HARDWARE DIAGNOSTICS")
print("=" * 60)
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    total_vram = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(f"CUDA Available: True")
    print(f"GPU Name:       {gpu_name}")
    print(f"Total VRAM:     {total_vram:.2f} GB")
    print(f"CUDA Version:   {torch.version.cuda}")
    print(f"PyTorch:        {torch.__version__}")
else:
    print("WARNING: No CUDA GPU detected. Check PyTorch installation!")
print("=" * 60)

In [ ]:
# Step 2: Configuration & Task Selection
import os

# Choose task: 'fakenews' or 'hatespeech'
TASK = 'fakenews'

# Model options: 'microsoft/deberta-v3-small', 'roberta-base', 'bert-base-uncased'
MODEL_NAME = 'microsoft/deberta-v3-small'

# 4GB VRAM Hyperparameters
BATCH_SIZE = 8
GRAD_ACCUM = 2       # Effective batch size = 16
MAX_LENGTH = 256     # Capped to fit within 4GB VRAM
EPOCHS = 3
LEARNING_RATE = 2e-5
FP16 = torch.cuda.is_available()

BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), '..'))
OUTPUT_DIR = os.path.join(BASE_DIR, 'models', 'deep_learning', TASK, MODEL_NAME.replace('/', '_'))
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Task:               {TASK}")
print(f"Model:              {MODEL_NAME}")
print(f"Batch Size:         {BATCH_SIZE} (Effective: {BATCH_SIZE * GRAD_ACCUM})")
print(f"Mixed Precision:    {FP16}")
print(f"Output Directory:   {OUTPUT_DIR}")

In [ ]:
# Step 3: Load Stratified Train, Val, and Test Splits
import pandas as pd
from datasets import Dataset, DatasetDict

splits_dir = os.path.join(BASE_DIR, 'data', 'splits')

if TASK == 'fakenews':
    train_df = pd.read_csv(os.path.join(splits_dir, 'train.csv'))
    val_df = pd.read_csv(os.path.join(splits_dir, 'val.csv'))
    test_df = pd.read_csv(os.path.join(splits_dir, 'test.csv'))
else:
    train_df = pd.read_csv(os.path.join(splits_dir, 'hate_train.csv'))
    val_df = pd.read_csv(os.path.join(splits_dir, 'hate_val.csv'))
    test_df = pd.read_csv(os.path.join(splits_dir, 'hate_test.csv'))

for df in [train_df, val_df, test_df]:
    if 'headline' in df.columns:
        df['text'] = df['headline'].fillna('') + " " + df['article_text'].fillna('')
    else:
        df['text'] = df['article_text'].fillna('')
    df['text'] = df['text'].astype(str).str.strip()
    df['label'] = df['label'].astype(int)

print(f"Train set: {len(train_df):,} rows")
print(f"Val set:   {len(val_df):,} rows")
print(f"Test set:  {len(test_df):,} rows")

raw_datasets = DatasetDict({
    'train': Dataset.from_pandas(train_df[['text', 'label']]),
    'val': Dataset.from_pandas(val_df[['text', 'label']]),
    'test': Dataset.from_pandas(test_df[['text', 'label']])
})

In [ ]:
# Step 4: Tokenization
from transformers import AutoTokenizer, DataCollatorWithPadding

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_function(examples):
    return tokenizer(examples['text'], truncation=True, max_length=MAX_LENGTH)

tokenized_datasets = raw_datasets.map(tokenize_function, batched=True, remove_columns=['text'])
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
print("Tokenization completed successfully.")

In [ ]:
# Step 5: Model Initialization & Metrics Function
import numpy as np
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

id2label = {0: "Real News" if TASK == 'fakenews' else "Safe",
            1: "Fake News" if TASK == 'fakenews' else "Hate Speech / Hostile"}
label2id = {v: k for k, v in id2label.items()}

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label=id2label,
    label2id=label2id
)

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    preds = np.argmax(predictions, axis=1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary')
    acc = accuracy_score(labels, preds)
    return {
        'accuracy': round(float(acc), 4),
        'f1': round(float(f1), 4),
        'precision': round(float(precision), 4),
        'recall': round(float(recall), 4)
    }

training_args = TrainingArguments(
    output_dir=os.path.join(OUTPUT_DIR, "checkpoints"),
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE * 2,
    gradient_accumulation_steps=GRAD_ACCUM,
    num_train_epochs=EPOCHS,
    weight_decay=0.01,
    warmup_ratio=0.1,
    fp16=FP16,
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    save_total_limit=1,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["val"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)
print("Model and Trainer initialized.")

In [ ]:
# Step 6: Start Training
print("Starting training on GPU...")
train_result = trainer.train()
print("Training finished!")

In [ ]:
# Step 7: Final Evaluation on Unseen Test Split
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

test_metrics = trainer.evaluate(tokenized_datasets["test"])
print("\nFinal Test Split Evaluation Results:")
for k, v in test_metrics.items():
    print(f"  {k}: {v}")

preds_raw = trainer.predict(tokenized_datasets["test"])
preds = np.argmax(preds_raw.predictions, axis=1)
labels = tokenized_datasets["test"]["label"]

cm = confusion_matrix(labels, preds)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=[id2label[0], id2label[1]],
            yticklabels=[id2label[0], id2label[1]])
plt.title(f"{MODEL_NAME} - Test Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "test_confusion_matrix.png"), dpi=300)
plt.show()

In [ ]:
# Step 8: Save Best Model Checkpoint
best_model_path = os.path.join(OUTPUT_DIR, "best_model")
trainer.save_model(best_model_path)
tokenizer.save_pretrained(best_model_path)
print(f"Model weights successfully saved to: {best_model_path}")